# 3-Level Duffing Verification — 5 ns Buffer Optimization

Imports the optimized controls from `robust_iswap_detuned_2MHz_130nsmw_5nsgauss_5nsbuf/`
(the 1k-iter 2-level 5ns-buffer iSWAP optimization), then simulates the same pulse against
the **3-level Duffing oscillator** model in the rotating frame (eq. (1) of
`Duffing_oscillator (18).pdf`).

## Model assumptions

- Both qubits at the same idle frequency ω_q (frames equal: ω₀¹ = ω₀² = ω_q)
- Drives also at ω_q (zero detuning Δ = 0, so the rotating-frame factor `exp(±i·ω₀·t)` on the drive is 1)
- Anharmonicity η = −2π · 0.170 rad/ns ≈ −170 MHz (transmon-style, negative)
- Coupling g_eff = 2π · 0.002 rad/ns = 2 MHz (constant during 130 ns microwave region, Gaussian-ramped during 5 ns rise/fall, square at g_eff during the 5 ns buffers on each side)
- 3 levels per mode (`|0⟩, |1⟩, |2⟩`); Hilbert-space dim = 9 = 3 × 3

## Rotating-frame Hamiltonian

From the PDF (eq. 1), with δᵢⱼ₀ = 0 (frames equal) and Δ_ℓ(t) = 0 (drives on resonance):

$$
H_I(t) \;=\; \sum_{\ell\in\{1,2\}} \Bigl[\,\delta\omega_\ell(t)\,\hat n_\ell \;+\; \tfrac{\eta}{2}\,a_\ell^{\dagger}a_\ell^{\dagger}a_\ell a_\ell \;+\; u_X^\ell(t)\,X_\ell \;+\; u_Y^\ell(t)\,Y_\ell\Bigr] \;+\; g(t)\,\bigl(X_1X_2 + Y_1Y_2\bigr)
$$

where `X_ℓ = a_ℓ + a_ℓ†`, `Y_ℓ = i(a_ℓ† − a_ℓ)`, `n̂_ℓ = a_ℓ† a_ℓ`.

- `g(t)·(X₁X₂ + Y₁Y₂) = 2g(t)·(a₁†a₂ + a₁a₂†)` — the standard RWA excitation-conserving exchange (counter-rotating `a₁†a₂† + h.c.` cancels in the X⊗X + Y⊗Y sum). The factor-of-2 matches the 2-level optimization convention `H_coupling = g_eff·(X⊗X + Y⊗Y)`.
- `δω_ℓ(t)` is the perturbation strength (frequency noise on each qubit). For the susceptibility sweep we set `δω_ℓ = ε` and sweep ε.
- `u_X, u_Y` are the optimized microwave envelopes (loaded from CSV); they are nonzero only inside the 130 ns microwave region.
- `g(t)` is the time-dependent coupling envelope: Gaussian rise (5 ns) → flat-at-g_eff buffer (5 ns) → flat-at-g_eff microwave region (130 ns) → flat buffer (5 ns) → Gaussian fall (5 ns), totalling 150 ns.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.add(["QuantumToolbox", "Optim"])
Pkg.instantiate()

using QuantumToolbox
using LinearAlgebra
using DelimitedFiles
using Printf
using CairoMakie
using Optim

# Operationally-correct fidelity: best F to U_target after a free single-qubit
# Z⊗Z phase rotation. This is what an experimentalist measures after calibrated
# virtual Z gates; it absorbs predictable coherent phase errors (e.g. the |11⟩
# AC-Stark shift from off-resonant coupling to |20⟩, |02⟩).
function best_F_with_virtual_Z(U_sub::AbstractMatrix, U_target::AbstractMatrix)
    obj = φ -> begin
        Zc = kron(Diagonal([1.0, exp(im*φ[1])]),
                  Diagonal([1.0, exp(im*φ[2])]))
        return -abs2(tr(U_target' * Zc * U_sub)) / 16
    end
    res = Optim.optimize(obj, [0.0, 0.0], NelderMead())
    return -Optim.minimum(res)
end

## 1. Physical parameters

Match the 2-level optimization (`robust_iswap_detuned_2MHz_150ns_5nsbuf.jl`) so the verification is apples-to-apples on the qubit subspace. The new knob is the anharmonicity η, which has no analog in the 2-level model.

In [ ]:
# Coupling between the two qubits (constant during the flat-top portion of g(t))
const g_eff = 2π * 0.002             # 2 MHz, rad/ns

# Anharmonicity per qubit. Convention: (η/2) a†a†aa = (η/2)·n̂·(n̂-1), so the |2⟩
# level shifts by η relative to the harmonic position 2ω_q. For transmons η < 0.
const η = -2π * 0.170                 # -170 MHz, rad/ns

# Edge and buffer parameters (must match the 5nsbuf optimization run)
const σ_rise              = 1.25      # ns ; 4σ = 5 ns Gaussian edge
const buffer_flat_duration = 5.0       # ns ; flat-at-g_eff buffer (microwaves OFF)
const buffer_duration     = 4 * σ_rise # ns ; Gaussian rise/fall width

# Total gate and microwave region
const T_total_gate_ns = 150.0
const T_mw_region_ns  = T_total_gate_ns - 2*buffer_duration - 2*buffer_flat_duration  # = 130 ns

# 3-level (qutrit) Duffing model
const n_levels = 3                     # 3 levels per mode
const Dim      = n_levels^2            # 9 = total Hilbert dim

println("g_eff = $(round(g_eff/(2π)*1e3, digits=1)) MHz   ($(g_eff) rad/ns)")
println("η     = $(round(η/(2π)*1e3,     digits=1)) MHz   ($(η) rad/ns)")
println("σ_rise = $σ_rise ns;  Gaussian edge = $(4*σ_rise) ns;  buffer = $buffer_flat_duration ns")
println("Total gate = $T_total_gate_ns ns;  microwave region = $T_mw_region_ns ns")
println("Hilbert dim = $Dim")

## 2. Load the optimized controls

`controls_fine.csv` has columns `time_ns, u_X1, u_Y1, u_X2, u_Y2` and contains 2000 evenly-spaced samples in the microwave region (0 → 130 ns). To place these on the absolute gate-frame time axis, we shift by `mw_start_t = buffer_duration + buffer_flat_duration = 10 ns`.

In [ ]:
const RUN_DIR  = joinpath(@__DIR__, "robust_iswap_detuned_2MHz_130nsmw_5nsgauss_5nsbuf")
const CSV_PATH = joinpath(RUN_DIR, "controls_fine.csv")
@assert isdir(RUN_DIR)   "Source run directory not found: $RUN_DIR"
@assert isfile(CSV_PATH) "controls_fine.csv not found in $RUN_DIR"

raw = readdlm(CSV_PATH, ',', skipstart=1)
const ts_mw  = vec(raw[:, 1])    # times within microwave region (≈ 0 → (N-1)/N · T_mw_nominal)
const uX1_mw = vec(raw[:, 2])
const uY1_mw = vec(raw[:, 3])
const uX2_mw = vec(raw[:, 4])
const uY2_mw = vec(raw[:, 5])

@assert isapprox(ts_mw[1], 0.0, atol = 1e-6)

# Piccolo's NamedTrajectory with N_knots and Δt = T_f / N_knots places knots at
#   t_k = (k-1)·Δt   for k = 1..N_knots
# So the LAST knot is at (N_knots - 1)·Δt = T_f · (N_knots - 1)/N_knots, NOT at T_f.
# With N_knots = 24 and T_f = 130 ns:  ts_mw[end] = 23/24 · 130 ≈ 124.58 ns.
# The verification has to use the ACTUAL evolved duration from the saved CSV — the
# nominal T_mw_region_ns is just the design parameter T_f.
const T_mw_actual = ts_mw[end]                         # actual microwave-region duration
const T_total_gate_actual = 2*buffer_duration + 2*buffer_flat_duration + T_mw_actual

println("Loaded $(length(ts_mw)) control samples")
@printf("  ts_mw spans [%.6f, %.6f] ns  (nominal T_mw = %.1f, ratio = %.4f)\n",
    ts_mw[1], ts_mw[end], T_mw_region_ns, ts_mw[end]/T_mw_region_ns)
@printf("  T_total_gate_actual = %.4f ns  (nominal = %.1f)\n",
    T_total_gate_actual, T_total_gate_ns)
for (lbl, arr) in zip(["u_X1","u_Y1","u_X2","u_Y2"], [uX1_mw, uY1_mw, uX2_mw, uY2_mw])
    @printf("  %s: min = %+.5f  max = %+.5f  rad/ns\n", lbl, minimum(arr), maximum(arr))
end

## 3. Build the full 150 ns pulse envelope (g(t) and microwaves)

`g(t)` has five segments: Gaussian rise (0..5 ns), flat buffer (5..10 ns), microwave-on flat (10..140 ns), flat buffer (140..145 ns), Gaussian fall (145..150 ns). Microwave envelopes are zero outside `[mw_start_t, mw_end_t] = [10, 140]` ns; inside, they are linearly interpolated from the loaded CSV (which uses the microwave-local time axis `[0, 130]`).

In [ ]:
const mw_start_t = buffer_duration + buffer_flat_duration       # 10.0 ns — first microwave sample
const mw_end_t   = mw_start_t + T_mw_actual                     # uses ACTUAL pulse duration (≠ 140 ns)

# g(t): coupling envelope on absolute time axis [0, T_total_gate_actual]
function g_envelope(t::Real)
    if t < buffer_duration
        # Gaussian rise: peak at t = buffer_duration
        return g_eff * exp(-(t - buffer_duration)^2 / (2*σ_rise^2))
    elseif t <= mw_end_t + buffer_flat_duration
        # Flat at g_eff through buffer + microwave + buffer
        return g_eff
    elseif t <= T_total_gate_actual
        # Gaussian fall: peak at t = mw_end_t + buffer_flat_duration
        t_post = t - (mw_end_t + buffer_flat_duration)
        return g_eff * exp(-t_post^2 / (2*σ_rise^2))
    else
        return 0.0
    end
end

# Microwave at absolute time t (zero outside microwave region; linear interp inside)
function microwave_at(arr::Vector{Float64}, t_abs::Real)
    if t_abs < mw_start_t || t_abs > mw_end_t
        return 0.0
    end
    t_local = t_abs - mw_start_t
    if t_local <= ts_mw[1];   return arr[1];   end
    if t_local >= ts_mw[end]; return arr[end]; end
    i = searchsortedlast(ts_mw, t_local)
    j = i + 1
    α = (t_local - ts_mw[i]) / (ts_mw[j] - ts_mw[i])
    return (1-α) * arr[i] + α * arr[j]
end

uX1(t) = microwave_at(uX1_mw, t)
uY1(t) = microwave_at(uY1_mw, t)
uX2(t) = microwave_at(uX2_mw, t)
uY2(t) = microwave_at(uY2_mw, t)
g_t(t) = g_envelope(t)

println("mw_start_t = $(mw_start_t) ns")
@printf("mw_end_t   = %.4f ns\n", mw_end_t)
@printf("Total gate (actual) = %.4f ns\n", T_total_gate_actual)

## 4. Plot the controls

Top panel: `g_eff(t)` envelope. Bottom panel: the four microwave channels.

In [ ]:
t_plot  = collect(0.0:0.05:T_total_gate_actual)
g_arr   = [g_t(t)  for t in t_plot]
uX1_arr = [uX1(t)  for t in t_plot]
uY1_arr = [uY1(t)  for t in t_plot]
uX2_arr = [uX2(t)  for t in t_plot]
uY2_arr = [uY2(t)  for t in t_plot]

fig = Figure(size = (1100, 600), fontsize = 20)

ax1 = Axis(fig[1, 1], ylabel = "g_eff(t)  [rad/ns]",
    title = @sprintf("Coupling envelope (total gate %.2f ns)", T_total_gate_actual))
lines!(ax1, t_plot, g_arr; linewidth = 2, color = :black)
vspan!(ax1, [0],                                [buffer_duration];                       color = (:gray, 0.10))
vspan!(ax1, [buffer_duration],                  [mw_start_t];                            color = (:goldenrod, 0.15))
vspan!(ax1, [mw_end_t],                         [mw_end_t + buffer_flat_duration];       color = (:goldenrod, 0.15))
vspan!(ax1, [mw_end_t + buffer_flat_duration],  [T_total_gate_actual];                   color = (:gray, 0.10))
hidexdecorations!(ax1, grid = false)

ax2 = Axis(fig[2, 1], xlabel = "t  [ns]", ylabel = "microwave amplitudes  [rad/ns]")
lines!(ax2, t_plot, uX1_arr; label = "u_X1", color = :crimson,     linewidth = 2)
lines!(ax2, t_plot, uY1_arr; label = "u_Y1", color = :orange,      linewidth = 2)
lines!(ax2, t_plot, uX2_arr; label = "u_X2", color = :forestgreen, linewidth = 2)
lines!(ax2, t_plot, uY2_arr; label = "u_Y2", color = :purple,      linewidth = 2)
axislegend(ax2; position = :rt)
vspan!(ax2, [0],                                [buffer_duration];                       color = (:gray, 0.10))
vspan!(ax2, [buffer_duration],                  [mw_start_t];                            color = (:goldenrod, 0.15))
vspan!(ax2, [mw_end_t],                         [mw_end_t + buffer_flat_duration];       color = (:goldenrod, 0.15))
vspan!(ax2, [mw_end_t + buffer_flat_duration],  [T_total_gate_actual];                   color = (:gray, 0.10))

linkxaxes!(ax1, ax2)
display(fig)

## 4b. Spectral content of the optimized drives

Power spectrum `|U(ω)|²` of the complex microwave envelope `u_X^ℓ(t) + i·u_Y^ℓ(t)` for each qubit. Whatever spectral weight the optimizer placed near the `1↔2` transition frequency `|η|/2π = 170 MHz` is what drives off-resonant `|1⟩→|2⟩` leakage. (First-order PT: leakage on qubit ℓ ≈ `2 · |U^ℓ(η)|²`.)

x-axis in MHz; vertical dashed line marks `|η|/2π`. y-axis is log-scaled so you can see content many decades below the DC peak.

In [ ]:
# Compute the Fourier transform of each qubit's complex microwave envelope and plot
# |U(ω)|² as a function of frequency in MHz. Done by direct DFT sum (no FFTW
# dependency); the sample count (length(ts_mw) ≈ 2000) is small enough that this is fast.

# Complex envelopes per qubit, sampled on the SAME grid as the loaded CSV
const u_q1_mw = uX1_mw .+ im .* uY1_mw     # length(ts_mw)
const u_q2_mw = uX2_mw .+ im .* uY2_mw
const dt_mw   = ts_mw[2] - ts_mw[1]

# Frequency grid: 0 → 400 MHz, in rad/ns internally
const f_MHz_grid = collect(0:0.5:400)                   # frequency axis in MHz
const ω_grid     = (2π / 1000) .* f_MHz_grid             # rad/ns

# Direct DFT: U(ω) = Σᵢ u(tᵢ) · exp(-i ω tᵢ) · dt
function dft_envelope(u_t, ts, ω)
    s = zero(ComplexF64)
    @inbounds for i in eachindex(ts)
        s += u_t[i] * exp(-im * ω * ts[i])
    end
    return s * (ts[2] - ts[1])
end

U_q1 = [dft_envelope(u_q1_mw, ts_mw, ω) for ω in ω_grid]
U_q2 = [dft_envelope(u_q2_mw, ts_mw, ω) for ω in ω_grid]

# Power spectrum in (rad/ns · ns)² = rad²; multiply by 2 for the √2 ladder element
# to get the rough leakage estimate per qubit.
P_q1 = abs2.(U_q1)
P_q2 = abs2.(U_q2)
leak_q1_est = 2 * P_q1[argmin(abs.(f_MHz_grid .- abs(η)/(2π) * 1000))]
leak_q2_est = 2 * P_q2[argmin(abs.(f_MHz_grid .- abs(η)/(2π) * 1000))]
@printf("First-order leakage estimate at |η|/2π = %.1f MHz:\n", abs(η)/(2π) * 1000)
@printf("  qubit 1 ≈ %.3e\n", leak_q1_est)
@printf("  qubit 2 ≈ %.3e\n", leak_q2_est)
@printf("  total   ≈ %.3e   (compare to measured leakage from cell 15)\n",
    leak_q1_est + leak_q2_est)

# Plot
fig = Figure(size = (1100, 500), fontsize = 20)
ax  = Axis(fig[1, 1];
    xlabel = "frequency  [MHz]",
    ylabel = "|U(ω)|²  [(rad)²]",
    title  = "Microwave envelope spectrum (log y)",
    yscale = log10,
)
lines!(ax, f_MHz_grid, max.(P_q1, 1e-16); label = "qubit 1",
    color = :crimson,    linewidth = 2)
lines!(ax, f_MHz_grid, max.(P_q2, 1e-16); label = "qubit 2",
    color = :forestgreen, linewidth = 2)
# Mark the |η|/2π anharmonicity frequency
η_MHz = abs(η) / (2π) * 1000
vlines!(ax, [η_MHz]; color = :black, linestyle = :dash, linewidth = 2)
text!(ax, η_MHz, maximum(P_q1) * 0.5; text = "|η|/2π ≈ $(round(η_MHz, digits=1)) MHz",
    align = (:left, :top), offset = (5, -5), fontsize = 16)
axislegend(ax; position = :rt)
ylims!(ax, 1e-10, 10 * maximum(vcat(P_q1, P_q2)))

display(fig)

# Save alongside the run-dir outputs
mkpath(joinpath(RUN_DIR, "figs"))
save(joinpath(RUN_DIR, "figs", "drive_spectrum.png"), fig)
println("Saved spectrum to: ", joinpath(RUN_DIR, "figs", "drive_spectrum.png"))

## 5. Build the 3-level Duffing operator basis (QuantumToolbox)

- `a_op = destroy(3)` — bosonic annihilation truncated at the 3rd level
- `tensor(a_op, I)` and `tensor(I, a_op)` to extend to the 2-qubit Hilbert space
- `X_ℓ = a_ℓ + a_ℓ†`, `Y_ℓ = i(a_ℓ† − a_ℓ)` (so that on the qubit subspace `Y|0⟩ = i|1⟩` matches σ_y)
- `n̂_ℓ = a_ℓ† a_ℓ` (number operator — for the dephasing perturbation directions)

The XX+YY combination equals `2(a₁†a₂ + a₁a₂†)`: the counter-rotating `a₁†a₂† + h.c.` parts cancel, leaving the RWA excitation-conserving exchange. The factor of 2 matches the 2-level optimization (where `g_eff · (XX+YY)` was used).

In [ ]:
# QuantumToolbox Qobj operators
a_op = destroy(n_levels)
I_op = qeye(n_levels)

a1 = tensor(a_op, I_op)            # annihilation on qubit 1
a2 = tensor(I_op, a_op)            # annihilation on qubit 2
a1d, a2d = a1', a2'                # creation operators

# Pauli analogs in 3-level (X, Y) and number operators (n̂)
X1op = a1 + a1d
Y1op = im * (a1d - a1)
X2op = a2 + a2d
Y2op = im * (a2d - a2)
n1op = a1d * a1
n2op = a2d * a2

# Anharmonicity per qubit: (η/2) a†a†aa = (η/2) n̂(n̂-1).
# Acts as 0 on |0⟩,|1⟩, and as η on |2⟩.
H_anh_op = (η/2) * (a1d * a1d * a1 * a1 + a2d * a2d * a2 * a2)

# XX+YY coupling structure (multiplied by g(t) at each time step below)
H_coupling_op = X1op * X2op + Y1op * Y2op

# Dense matrices for the propagation loop (faster + less Qobj overhead per step)
const H_coupling_mat = Matrix(H_coupling_op.data)
const H_anh_mat      = Matrix(H_anh_op.data)
const X1_mat = Matrix(X1op.data); const Y1_mat = Matrix(Y1op.data)
const X2_mat = Matrix(X2op.data); const Y2_mat = Matrix(Y2op.data)
const n1_mat = Matrix(n1op.data); const n2_mat = Matrix(n2op.data)
const nn_mat = n1_mat * n2_mat

println("Hilbert space dim: ", size(H_coupling_mat, 1), "  (expected $(Dim))")
println("H_anh diagonal (rad/ns): ", round.(real(diag(H_anh_mat)), digits=3))

## 6. Time-dependent Hamiltonian + piecewise-constant propagator

`H_total(t, ε, H_err_mat)` returns the 9×9 Hamiltonian at time t in the rotating frame. The perturbation acts for the **entire 150 ns** (during edges, buffers, and microwave region).

`propagate(H_func, t_grid)` builds `U_T = ∏_k exp(−i·H(t_mid_k)·Δt_k)` using midpoint sampling. Step size 50 ps gives ~3000 steps over 150 ns — fast enough for 51-point ε sweeps, accurate enough that the residual from piecewise-constant approximation is well below the leakage floor.

In [ ]:
function H_total(t::Real, ε::Real = 0.0, H_err_mat = nothing)
    g = g_t(t)
    H = g * H_coupling_mat +                       # coupling (XX+YY) · g(t)
        H_anh_mat +                                # anharmonicity (η/2) n̂(n̂-1)
        uX1(t) * X1_mat + uY1(t) * Y1_mat +        # qubit 1 drives (rotating frame, on resonance)
        uX2(t) * X2_mat + uY2(t) * Y2_mat          # qubit 2 drives
    if H_err_mat !== nothing && ε != 0.0
        H += ε * H_err_mat                         # perturbation (δω·n̂ or n̂₁n̂₂)
    end
    return H
end

function propagate(H_func, t_grid::Vector{Float64})
    U = Matrix{ComplexF64}(I, Dim, Dim)
    @inbounds for i in 1:length(t_grid)-1
        dt   = t_grid[i+1] - t_grid[i]
        tmid = (t_grid[i] + t_grid[i+1]) / 2
        H    = H_func(tmid)
        U    = exp(-im * dt * H) * U
    end
    return U
end

const dt_sim     = 0.05    # ns; 50 ps step
const t_grid_sim = collect(0:dt_sim:T_total_gate_actual)   # use the ACTUAL gate duration
println("Propagator grid: ", length(t_grid_sim), " points  (dt = $dt_sim ns)")
@printf("Grid spans 0.0 → %.4f ns (actual gate)\n", t_grid_sim[end])

## 7. Propagate without perturbation — verify the 3-level fidelity

Propagate the optimized pulse through the 3-level Duffing model with ε = 0. Project the final 9×9 unitary onto the computational subspace (the 4 states `|i₁⟩⊗|i₂⟩` with `i₁,i₂ ∈ {0,1}` — indices `[1, 2, 4, 5]` in the 3⊗3 basis ordering) and compute fidelity to iSWAP.

Two metrics:
- **Subspace fidelity** `F = |Tr(U_iSWAP† · P U P)|² / d²` where P is the projector onto the computational subspace.
- **Leakage** `1 − Tr(PU† PU P) / d` — population that has left the qubit subspace.

In [ ]:
# Propagate through full 150 ns with no perturbation
U_T = propagate(t -> H_total(t, 0.0), t_grid_sim)
@printf("Unitarity check: ‖U†U − I‖ = %.2e\n", norm(U_T'*U_T - I))

# Computational subspace (3⊗3 basis order |i₁i₂⟩ with i = i₁·3 + i₂ + 1)
#   |00⟩ → 1,  |01⟩ → 2,  |10⟩ → 4,  |11⟩ → 5
const subspace = [1, 2, 4, 5]
U_T_sub = U_T[subspace, subspace]

# Target = iSWAP in qubit basis
const U_iswap_4x4 = let
    σx = ComplexF64[0.0 1.0; 1.0 0.0]
    σy = ComplexF64[0.0 -im; im 0.0]
    exp(-im * (π/4) * (kron(σx, σx) + kron(σy, σy)))
end

F_sub = abs2(tr(U_iswap_4x4' * U_T_sub)) / 16
leak  = 1 - real(tr(U_T_sub' * U_T_sub)) / 4
@printf("3-level subspace fidelity to iSWAP:  %.6f   (infidelity %.3e)\n", F_sub, 1 - F_sub)
@printf("Leakage out of computational subspace: %.3e\n", leak)

## 8. Susceptibility sweep — fidelity vs. perturbation strength ε

Three perturbation directions, each a static `H_err` added to the rotating-frame Hamiltonian for the full 150 ns:

- `n̂_1` — frequency noise on qubit 1 (dephasing)
- `n̂_2` — frequency noise on qubit 2
- `n̂_1 · n̂_2` — common-mode product (the "ZZ-equivalent" in the n̂ basis)

Note on equivalence to the 2-level `Z` basis: on the qubit subspace `n̂_ℓ = (I − Z_ℓ)/2`, so `ε·n̂_ℓ` ≡ `−(ε/2)·Z_ℓ` up to a global phase. To compare ε-axis directly with the 2-level Z-noise sweep, multiply this ε by 2 (or use `2·n̂_ℓ` as the perturbation operator).

In [ ]:
const errors = [
    ("n1",   n1_mat),
    ("n2",   n2_mat),
    ("n1n2", nn_mat),
]
const εs = collect(range(-0.02, 0.02, length = 51))  # rad/ns

F_vs_eps    = Dict{String, Vector{Float64}}()
leak_vs_eps = Dict{String, Vector{Float64}}()

for (name, H_err) in errors
    F_arr    = Float64[]
    leak_arr = Float64[]
    for ε in εs
        U_ε     = propagate(t -> H_total(t, ε, H_err), t_grid_sim)
        U_ε_sub = U_ε[subspace, subspace]
        push!(F_arr,    abs2(tr(U_iswap_4x4' * U_ε_sub)) / 16)
        push!(leak_arr, 1 - real(tr(U_ε_sub' * U_ε_sub)) / 4)
    end
    F_vs_eps[name]    = F_arr
    leak_vs_eps[name] = leak_arr
    @printf("  %-6s done — F(ε=0) = %.5f, max leak = %.2e\n", name, F_arr[length(εs)÷2+1], maximum(leak_arr))
end

## 8b. Default Gaussian-square baseline (coupling-only, no microwaves)

The "default" pulse is the simplest no-optimization iSwap implementation: same Gaussian
edge shape (4σ = 5 ns) and the SAME coupling g_eff, but the flat-top width is chosen so
that the **total integrated area equals π/4** — i.e., the bare coupling drives the
iSwap rotation without any microwave compensation.

There is **no buffer region** in the default (microwaves are simply absent, so there's
nothing to buffer from). Total gate is `4σ + flat_default + 4σ`, shorter than 150 ns.

Propagating this through the 3-level Duffing model lets us check whether the robust
optimization actually helps in the 3-level world. In 2 levels the default is exact iSWAP;
in 3 levels we'll see how much the `|11⟩ ↔ |20⟩, |02⟩` coupling-induced leakage and the
edge-time leakage cost the default.

In [ ]:
# Build the default Gaussian-square pulse: Gauss rise + flat at g_eff + Gauss fall.
# Choose flat_default so that the total area equals π/4 (full iSWAP rotation).

function gaussian_square_area(flat_ns; gv = g_eff, σv = σ_rise, dtv = 0.01, trunc = 4)
    buf = trunc * σv
    t_rise = collect(0:dtv:buf)
    g_rise_seg = gv .* exp.(-(t_rise .- buf).^2 ./ (2 * σv^2))
    A_rise = sum(g_rise_seg) * dtv
    A_flat = gv * flat_ns
    A_fall = A_rise
    return A_rise + A_flat + A_fall
end

# Bisect for the flat-top width that gives total area = π/4
const flat_default = let target = π/4, lo = 0.0, hi = 400.0
    for _ in 1:100
        mid = (lo + hi) / 2
        gaussian_square_area(mid) < target ? (lo = mid) : (hi = mid)
    end
    (lo + hi) / 2
end

const T_default = buffer_duration + flat_default + buffer_duration

@printf("Default Gaussian-square flat-top width = %.4f ns\n", flat_default)
@printf("Default total gate time = %.4f ns\n", T_default)
@printf("Verification: total area = %.6f rad   (target π/4 = %.6f rad)\n",
    gaussian_square_area(flat_default), π/4)

# g_default(t): coupling envelope for the default pulse (NO microwaves anywhere)
function g_default_envelope(t::Real)
    if t < buffer_duration
        return g_eff * exp(-(t - buffer_duration)^2 / (2 * σ_rise^2))
    elseif t <= buffer_duration + flat_default
        return g_eff
    elseif t <= T_default
        t_post = t - (buffer_duration + flat_default)
        return g_eff * exp(-t_post^2 / (2 * σ_rise^2))
    else
        return 0.0
    end
end

# Hamiltonian for the default pulse (no microwaves; same anharmonicity + coupling form)
function H_default(t::Real, ε::Real = 0.0, H_err_mat = nothing)
    g = g_default_envelope(t)
    H = g * H_coupling_mat + H_anh_mat
    if H_err_mat !== nothing && ε != 0.0
        H += ε * H_err_mat
    end
    return H
end

# Propagator grid for the default — shorter than the robust pulse
const t_grid_default = collect(0:dt_sim:T_default)
println("Default propagator grid: ", length(t_grid_default), " points")

# Sanity check: propagate default with no perturbation, report subspace fidelity + leakage
U_def_0     = propagate(t -> H_default(t, 0.0), t_grid_default)
U_def_0_sub = U_def_0[subspace, subspace]
F_def_0     = abs2(tr(U_iswap_4x4' * U_def_0_sub)) / 16
leak_def_0  = 1 - real(tr(U_def_0_sub' * U_def_0_sub)) / 4
@printf("Default (no perturbation):  F_sub = %.6f   (infid %.3e)   leak = %.3e\n",
    F_def_0, 1 - F_def_0, leak_def_0)

In [ ]:
# Default susceptibility sweep: same ε grid + same error operators as the robust sweep,
# but propagating through the SHORT default pulse (no microwaves, no buffers).

F_default_vs_eps    = Dict{String, Vector{Float64}}()
leak_default_vs_eps = Dict{String, Vector{Float64}}()

for (name, H_err) in errors
    F_arr    = Float64[]
    leak_arr = Float64[]
    for ε in εs
        U_ε     = propagate(t -> H_default(t, ε, H_err), t_grid_default)
        U_ε_sub = U_ε[subspace, subspace]
        push!(F_arr,    abs2(tr(U_iswap_4x4' * U_ε_sub)) / 16)
        push!(leak_arr, 1 - real(tr(U_ε_sub' * U_ε_sub)) / 4)
    end
    F_default_vs_eps[name]    = F_arr
    leak_default_vs_eps[name] = leak_arr
    @printf("  default %-6s done — F(ε=0) = %.5f, max leak = %.2e\n",
        name, F_arr[length(εs)÷2+1], maximum(leak_arr))
end

## 9. Plot the susceptibility curves

x-axis in **MHz** (consistent with the 2-level notebooks): `ε [MHz] = ε [rad/ns] · 1000 / (2π)`.

In [ ]:
εs_MHz = εs .* 1000 ./ (2π)

fig = Figure(size = (1500, 600), fontsize = 20)
for (col, name) in enumerate(["n1", "n2", "n1n2"])
    ax = Axis(fig[1, col]; xlabel = "ε  [MHz]", ylabel = "1 − F",
        title = "Perturbation: $name (3-level Duffing)", yscale = log10)
    # Robust — solid red (infidelity), dashed red (leakage)
    lines!(ax, εs_MHz, max.(1 .- F_vs_eps[name], 1e-16);
        linewidth = 2, color = :crimson,  label = "robust  (1−F)")
    lines!(ax, εs_MHz, max.(leak_vs_eps[name], 1e-16);
        linewidth = 2, color = :crimson,  linestyle = :dash, label = "robust  (leak)")
    # Default — solid black (infidelity), dashed black (leakage)
    lines!(ax, εs_MHz, max.(1 .- F_default_vs_eps[name], 1e-16);
        linewidth = 2, color = :black,    label = "default (1−F)")
    lines!(ax, εs_MHz, max.(leak_default_vs_eps[name], 1e-16);
        linewidth = 2, color = :black,    linestyle = :dash, label = "default (leak)")
    ylims!(ax, 1e-5, 1.0)
    col == 1 && axislegend(ax; position = :lb, labelsize = 14)
end
display(fig)

outdir = joinpath(@__DIR__, basename(RUN_DIR))
mkpath(joinpath(outdir, "figs"))
save(joinpath(outdir, "figs", "duffing_3level_susceptibility_vs_default.png"), fig)
println("Saved susceptibility figure to: ",
    joinpath(outdir, "figs", "duffing_3level_susceptibility_vs_default.png"))

## 10. Bandwidth-filter test — does a 250 MHz AWG filter kill the leakage?

The 2-level optimization is free to place spectral content at `|η|/2π = 170 MHz`, which off-resonantly drives `|1⟩→|2⟩` in the 3-level model. On the hardware, the AWG has a finite bandwidth — we model it as a Gaussian filter with `B_3dB = 250 MHz` (the convention in `CLAUDE.md`).

If filtering the optimized 2-level pulse through this hardware filter suppresses the 170 MHz spectral content enough to kill leakage WITHOUT killing in-band fidelity, then we don't need to re-optimize — the hardware already does the work. If it doesn't, we know we need either a tighter filter or true 3-level optimization.

Filter is applied to the complex baseband `u_X + i·u_Y` per qubit; transfer function `G(f) = exp(−f²/(2σ_f²))` with `σ_f = B_3dB / √(ln 2)`.

In [ ]:
using FFTW

const B_3dB_MHz = 250.0
const σ_f_MHz   = B_3dB_MHz / sqrt(log(2))

# Sample grid from the loaded controls (CSV samples are uniformly spaced)
const N_mw     = length(ts_mw)
const fs_GHz   = 1.0 / dt_mw
const freqs_MHz = fftfreq(N_mw, fs_GHz) .* 1000.0

# Gaussian transfer function
const Gfilt = exp.(-(freqs_MHz.^2) ./ (2 * σ_f_MHz^2))

# Apply to complex baseband u_X + i·u_Y per qubit
function apply_bw_filter(uX, uY)
    z      = uX .+ im .* uY
    z_filt = ifft(fft(z) .* Gfilt)
    return real.(z_filt), imag.(z_filt)
end

uX1_filt_mw, uY1_filt_mw = apply_bw_filter(uX1_mw, uY1_mw)
uX2_filt_mw, uY2_filt_mw = apply_bw_filter(uX2_mw, uY2_mw)

# Time-domain comparison
fig = Figure(size = (1200, 700), fontsize = 18)
for (i, (lbl, raw, filt)) in enumerate([
        ("u_X1", uX1_mw, uX1_filt_mw),
        ("u_Y1", uY1_mw, uY1_filt_mw),
        ("u_X2", uX2_mw, uX2_filt_mw),
        ("u_Y2", uY2_mw, uY2_filt_mw),
    ])
    ax = Axis(fig[i, 1], ylabel = lbl)
    lines!(ax, ts_mw, raw;  color = :gray,    linewidth = 2, label = "raw")
    lines!(ax, ts_mw, filt; color = :crimson, linewidth = 2,
        label = "filtered ($(Int(B_3dB_MHz)) MHz)")
    i == 1 && axislegend(ax; position = :rt)
    i < 4  && hidexdecorations!(ax, grid = false)
end
Label(fig[5, 1], "t in microwave region [ns]", padding = (0, 0, 0, 5))
display(fig)

# Spectrum comparison (qubit 1 only — qubit 2 is similar)
fig2 = Figure(size = (1100, 500), fontsize = 20)
ax = Axis(fig2[1, 1]; xlabel = "frequency [MHz]", ylabel = "|U(ω)|²",
    title = "Filtered vs raw spectrum (qubit 1, complex baseband)", yscale = log10)
z1   = uX1_mw      .+ im .* uY1_mw
z1f  = uX1_filt_mw .+ im .* uY1_filt_mw
P1   = abs2.(fft(z1))  .* dt_mw^2
P1f  = abs2.(fft(z1f)) .* dt_mw^2
pos  = freqs_MHz .>= 0
lines!(ax, freqs_MHz[pos], max.(P1[pos],  1e-16);
    color = :gray,    linewidth = 2, label = "raw")
lines!(ax, freqs_MHz[pos], max.(P1f[pos], 1e-16);
    color = :crimson, linewidth = 2, label = "filtered")
η_MHz = abs(η) / (2π) * 1000
vlines!(ax, [η_MHz];     color = :black, linestyle = :dash, linewidth = 2)
vlines!(ax, [B_3dB_MHz]; color = :blue,  linestyle = :dot,  linewidth = 2)
xlims!(ax, 0, 600)
ylims!(ax, 1e-12, maximum(P1) * 10)
axislegend(ax; position = :rt)
display(fig2)

idx_η = argmin(abs.(freqs_MHz .- η_MHz))
@printf("Spectral power at |η|/2π = %.1f MHz (qubit 1):\n", η_MHz)
@printf("  raw      |U(η)|² = %.3e\n", P1[idx_η])
@printf("  filtered |U(η)|² = %.3e  (ratio %.2e)\n",
    P1f[idx_η], P1f[idx_η] / P1[idx_η])

In [ ]:
# Filtered interpolators (linear interp on the same ts_mw grid as the raw controls)
function microwave_at_filt(arr_filt, t_abs)
    (t_abs < mw_start_t || t_abs > mw_end_t) && return 0.0
    t_local = t_abs - mw_start_t
    t_local <= ts_mw[1]   && return arr_filt[1]
    t_local >= ts_mw[end] && return arr_filt[end]
    i = searchsortedlast(ts_mw, t_local)
    α = (t_local - ts_mw[i]) / (ts_mw[i+1] - ts_mw[i])
    return (1-α) * arr_filt[i] + α * arr_filt[i+1]
end

uX1_f(t) = microwave_at_filt(uX1_filt_mw, t)
uY1_f(t) = microwave_at_filt(uY1_filt_mw, t)
uX2_f(t) = microwave_at_filt(uX2_filt_mw, t)
uY2_f(t) = microwave_at_filt(uY2_filt_mw, t)

function H_total_filt(t::Real, ε::Real = 0.0, H_err_mat = nothing)
    g = g_t(t)
    H = g * H_coupling_mat + H_anh_mat +
        uX1_f(t) * X1_mat + uY1_f(t) * Y1_mat +
        uX2_f(t) * X2_mat + uY2_f(t) * Y2_mat
    if H_err_mat !== nothing && ε != 0.0
        H += ε * H_err_mat
    end
    return H
end

# Sanity propagation at ε = 0
U_T_f     = propagate(t -> H_total_filt(t, 0.0), t_grid_sim)
U_T_f_sub = U_T_f[subspace, subspace]
F_f       = abs2(tr(U_iswap_4x4' * U_T_f_sub)) / 16
leak_f    = 1 - real(tr(U_T_f_sub' * U_T_f_sub)) / 4
@printf("Filtered (B_3dB = %.0f MHz, no perturbation):\n", B_3dB_MHz)
@printf("  F_sub = %.6f   (infid %.3e)   leak = %.3e\n", F_f, 1 - F_f, leak_f)
@printf("Compare raw (no filter): F_sub = %.6f, leak = %.3e\n", F_sub, leak)

# Susceptibility sweep with filtered pulse
F_filt_vs_eps    = Dict{String, Vector{Float64}}()
leak_filt_vs_eps = Dict{String, Vector{Float64}}()
for (name, H_err) in errors
    F_arr    = Float64[]
    leak_arr = Float64[]
    for ε in εs
        U_ε  = propagate(t -> H_total_filt(t, ε, H_err), t_grid_sim)
        Usub = U_ε[subspace, subspace]
        push!(F_arr,    abs2(tr(U_iswap_4x4' * Usub)) / 16)
        push!(leak_arr, 1 - real(tr(Usub' * Usub)) / 4)
    end
    F_filt_vs_eps[name]    = F_arr
    leak_filt_vs_eps[name] = leak_arr
    @printf("  filt %-6s done — F(ε=0) = %.5f, max leak = %.2e\n",
        name, F_arr[length(εs)÷2+1], maximum(leak_arr))
end

# Compare filtered / raw robust / default for each error direction
fig = Figure(size = (1500, 600), fontsize = 20)
for (col, name) in enumerate(["n1", "n2", "n1n2"])
    ax = Axis(fig[1, col]; xlabel = "ε  [MHz]", ylabel = "1 − F",
        title = "$name   ($(Int(B_3dB_MHz)) MHz filter)", yscale = log10)
    lines!(ax, εs_MHz, max.(1 .- F_vs_eps[name], 1e-16);
        color = :gray,    linewidth = 2, label = "raw robust  (1−F)")
    lines!(ax, εs_MHz, max.(leak_vs_eps[name], 1e-16);
        color = :gray,    linewidth = 2, linestyle = :dash)
    lines!(ax, εs_MHz, max.(1 .- F_filt_vs_eps[name], 1e-16);
        color = :crimson, linewidth = 2, label = "filtered  (1−F)")
    lines!(ax, εs_MHz, max.(leak_filt_vs_eps[name], 1e-16);
        color = :crimson, linewidth = 2, linestyle = :dash)
    lines!(ax, εs_MHz, max.(1 .- F_default_vs_eps[name], 1e-16);
        color = :black,   linewidth = 2, label = "default  (1−F)")
    lines!(ax, εs_MHz, max.(leak_default_vs_eps[name], 1e-16);
        color = :black,   linewidth = 2, linestyle = :dash)
    ylims!(ax, 1e-5, 1.0)
    col == 1 && axislegend(ax; position = :lb, labelsize = 14)
end
display(fig)

outdir = joinpath(@__DIR__, basename(RUN_DIR))
mkpath(joinpath(outdir, "figs"))
save(joinpath(outdir, "figs",
    @sprintf("duffing_3level_susceptibility_filtered_%dMHz.png", Int(B_3dB_MHz))), fig)
println("Saved filtered comparison.")